In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: GEO02 - Business Travel
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. Cost Center Mapping: vw_cost_center_mapping_bootstrap
   3. Country Risk Ratings: hive_metastore.ra_adido_2025.td_country_risk_rating_abac
   4. AMEX Travel Data: hive_metastore.ra_adido_2025.compliance_country_destination_report_amex_gbt_aug12_oct31_25
   5. CWT Travel Data: hive_metastore.ra_adido_2025.compliance_country_destination_report_cwt_nov24_aug11_25
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. HIGH RISK JURISDICTIONS: Queries the ABAC risk table for 'High' ratings.
   3. TRAVEL DATA UNION: Unions AMEX and CWT files, extracting standard cost center 
      integers and uppercase destination countries.
   4. FILTERING: INNER JOINS Travel Data to the High Risk Countries list.
   5. MAPPING & AGGREGATING: Joins high-risk trips to the CC Mapping table, 
      then SUMS the total number of trips per AU_ID.
   6. FINAL OUTPUT: Strict 6-column structure. LEFT JOINS mapped trips onto the 
      Master AU anchor, converting NULL sums to '0'.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

High_Risk_Countries AS (
    -- 2. Grab high-risk jurisdictions
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE UPPER(TRIM(FinalABACRating)) = 'HIGH'
),

Combined_Travel_Data AS (
    -- 3a. AMEX data
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        TRY_CAST(TRIM(Numberoftrips) AS INT) AS Trip_Count,
        'AMEX' AS Source_System
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_amex_gbt_aug12_oct31_25
    
    UNION ALL
    
    -- 3b. CWT data
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        1 AS Trip_Count,
        'CWT' AS Source_System
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_cwt_nov24_aug11_25
),

High_Risk_Trips AS (
    -- 4. Keep only high-risk destinations
    SELECT 
        t.Cleaned_CC,
        t.DestinationCountry,
        t.Trip_Count,
        t.Source_System
    FROM Combined_Travel_Data t
    INNER JOIN High_Risk_Countries h
        ON t.DestinationCountry = h.CountryName
    WHERE t.Cleaned_CC IS NOT NULL
),

CC_Mapping AS (
    -- Standardize the CC Mapping table using the correct cost_center_id column
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Trips_By_AU AS (
    -- 5. Map trips to AU_ID and sum
    SELECT 
        m.AU_ID,
        SUM(h.Trip_Count) AS Total_Trips
    FROM High_Risk_Trips h
    INNER JOIN CC_Mapping m
        ON h.Cleaned_CC = m.Mapped_CC
    GROUP BY m.AU_ID
)

-- 6. Final Template
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'GEO02' AS QuestionID,               
    COALESCE(CAST(t.Total_Trips AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Trips_By_AU t 
    ON a.BusinessID = t.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: GEO02 Drill-Down
   
   PURPOSE: Shows EVERY high-risk travel record from AMEX and CWT, regardless of 
   whether the Cost Center maps to an AU, or whether that AU exists in the Master List.
   Useful for tracking rogue travel mapped to orphaned Cost Centers.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE UPPER(TRIM(FinalABACRating)) = 'HIGH'
),

Combined_Travel_Data AS (
    SELECT 
        CostCenter AS Raw_String,
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        TRY_CAST(TRIM(Numberoftrips) AS INT) AS Trip_Count,
        'AMEX' AS Source_System
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_amex_gbt_aug12_oct31_25
    UNION ALL
    SELECT 
        CostCenter AS Raw_String,
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        1 AS Trip_Count,
        'CWT' AS Source_System
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_cwt_nov24_aug11_25
),

High_Risk_Trips AS (
    SELECT 
        t.Raw_String,
        t.Cleaned_CC,
        t.DestinationCountry,
        t.Trip_Count,
        t.Source_System
    FROM Combined_Travel_Data t
    INNER JOIN High_Risk_Countries h
        ON t.DestinationCountry = h.CountryName
    WHERE t.Cleaned_CC IS NOT NULL
),

CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
)

SELECT 
    COALESCE(m.AU_ID, 'UNMAPPED_CC') AS BusinessID,
    COALESCE(mast.In_ABAC_AU_List, 'No') AS In_ABAC_AU_List,
    h.Cleaned_CC AS Flagged_Cost_Center,
    h.DestinationCountry AS High_Risk_Destination,
    h.Trip_Count,
    h.Source_System,
    h.Raw_String AS Original_Cost_Center_Data
FROM High_Risk_Trips h
LEFT JOIN CC_Mapping m
    ON h.Cleaned_CC = m.Mapped_CC
LEFT JOIN Master_AUs mast
    ON m.AU_ID = mast.BusinessID
ORDER BY BusinessID, h.Cleaned_CC;